# LeetCode 3931 — Check Adjacent Digit Differences bằng Python

Link đề: https://leetcode.com/problems/check-adjacent-digit-differences/ · Độ khó: **Easy** · Chủ đề: String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def isAdjacentDiffAtMostTwo(self, s: str) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi chữ số `s` thỏa mãn khi **mọi** cặp chữ số liền kề có `|a - b| <= 2`.
2. Biết chuyển ký tự chữ số thành số nguyên (`int(c)`, `ord(c) - ord('0')`) và duyệt cặp liền kề bằng chỉ số hoặc `zip(s, s[1:])`.
3. Nắm mẫu **kiểm tra điều kiện với mọi phần tử** (`all`, thoát sớm khi gặp phần tử vi phạm).
4. Viết 4 cách khác nhau (vòng lặp, `zip + all`, số học `ord`, bảng tra cứu) và đối chiếu chúng bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi hay gặp: quên `abs`, nhầm `<` với `<=`, lệch chỉ số `i + 1`, trừ trực tiếp hai ký tự.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các chữ số `'0'..'9'`. Hãy trả về `True` nếu **hiệu tuyệt đối của mọi cặp chữ số đứng cạnh nhau** đều **không vượt quá 2**, ngược lại trả về `False`. Hiệu tuyệt đối của `a` và `b` là `abs(a - b)`.

| Ví dụ | `s` | Output | Giải thích |
|---|---|---|---|
| 1 | `"132"` | `True` | `abs(1-3) = 2 <= 2`, `abs(3-2) = 1 <= 2` |
| 2 | `"129"` | `False` | `abs(1-2) = 1` ổn, nhưng `abs(2-9) = 7 > 2` |

Constraints: `2 <= len(s) <= 100`; `s` chỉ gồm chữ số.

Lưu ý quan trọng: "liền kề" nghĩa là **hai vị trí cạnh nhau** `(s[i], s[i+1])`, không phải mọi cặp bất kỳ. Chuỗi dài `n` có đúng `n - 1` cặp liền kề.

In [1]:
s = "132"
for i in range(len(s) - 1):
    a, b = int(s[i]), int(s[i + 1])
    print(f"cặp ({a}, {b}): |{a}-{b}| = {abs(a - b)}  ->  {'ổn' if abs(a - b) <= 2 else 'VI PHẠM'}")
print("---")
s = "129"
for i in range(len(s) - 1):
    a, b = int(s[i]), int(s[i + 1])
    print(f"cặp ({a}, {b}): |{a}-{b}| = {abs(a - b)}  ->  {'ổn' if abs(a - b) <= 2 else 'VI PHẠM'}")

cặp (1, 3): |1-3| = 2  ->  ổn
cặp (3, 2): |3-2| = 1  ->  ổn
---
cặp (1, 2): |1-2| = 1  ->  ổn
cặp (2, 9): |2-9| = 7  ->  VI PHẠM


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự, chỉ số bắt đầu từ 0

`s[i]` là ký tự thứ `i`. `len(s)` là độ dài. Chỉ số hợp lệ là `0 .. len(s)-1`. Với chuỗi `"132"`: `s[0] = '1'`, `s[1] = '3'`, `s[2] = '2'`. Truy cập `s[3]` gây `IndexError`.

In [2]:
s = "132"
print(len(s), s[0], s[1], s[2])
try:
    print(s[3])
except IndexError as e:
    print("IndexError:", e)

3 1 3 2
IndexError: string index out of range


### 3.2. Ký tự chữ số khác số nguyên

`'3'` là **chuỗi** có độ dài 1; `3` là **số**. Không thể trừ hai chuỗi: `'3' - '1'` gây `TypeError`. Hai cách đổi thông dụng:

- `int('3')` cho `3` (rõ ràng, dễ đọc);
- `ord('3') - ord('0')` cho `3` (dùng mã ASCII: các chữ số `'0'..'9'` có mã liên tiếp `48..57`).

In [3]:
print(int('7'), ord('7') - ord('0'), ord('0'), ord('9'))
try:
    print('3' - '1')
except TypeError as e:
    print("TypeError:", e)

7 7 48 57
TypeError: unsupported operand type(s) for -: 'str' and 'str'


### 3.3. `abs` và so sánh `<=`

`abs(x)` là giá trị tuyệt đối: `abs(-5) = 5`. Hiệu `a - b` có thể âm, nên cần `abs`, nếu không `1 - 9 = -8` sẽ bị coi là "nhỏ hơn 2". Phép so sánh phải là **`<= 2`** (cho phép đúng bằng 2), không phải `< 2`.

In [4]:
print(abs(-5), abs(5), 1 - 9, abs(1 - 9))
print(2 <= 2, 2 < 2)

5 5 -8 8
True False


### 3.4. `zip(s, s[1:])` để lấy từng cặp liền kề

`s[1:]` là chuỗi bỏ ký tự đầu. `zip(s, s[1:])` ghép phần tử cùng vị trí: `(s[0], s[1])`, `(s[1], s[2])`, ... và dừng ở dãy ngắn hơn, nên cho đúng `n - 1` cặp, không cần lo chỉ số.

In [5]:
s = "1329"
print(list(zip(s, s[1:])))

[('1', '3'), ('3', '2'), ('2', '9')]


### 3.5. `all(...)` và `any(...)`

`all(iterable)` trả `True` nếu **mọi** phần tử đều đúng (và dừng ngay khi gặp một phần tử sai: *short-circuit*). `any(iterable)` trả `True` nếu **ít nhất một** phần tử đúng. Với iterable rỗng, `all([]) = True` và `any([]) = False`.

In [6]:
print(all([True, True]), all([True, False]), all([]))
print(any([False, True]), any([False, False]), any([]))
print(all(x < 3 for x in [1, 2, 3, 99]))   # dừng ở 3, không xét 99

True False True
True False False
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy hình dung các chữ số là các điểm trên **trục số 0..9**. Đi từ chữ số này sang chữ số kế tiếp, ta "nhảy" một đoạn. Điều kiện của đề nói: **mỗi bước nhảy dài tối đa 2**. Chỉ cần **một** bước nhảy dài hơn 2 là cả chuỗi hỏng.

Tính tay `"4657"`: `4->6` nhảy 2 (ổn), `6->5` nhảy 1 (ổn), `5->7` nhảy 2 (ổn) -> `True`.
Tính tay `"4690"`: `4->6` ổn, `6->9` nhảy 3 (hỏng) -> `False`, không cần xét tiếp `9->0`.

| Chuỗi | Các bước nhảy | Kết quả |
|---|---|---|
| `"132"` | 2, 1 | `True` |
| `"129"` | 1, 7 | `False` |
| `"4657"` | 2, 1, 2 | `True` |
| `"4690"` | 2, 3, 9 | `False` |

In [7]:
def jumps(s):
    return [abs(int(a) - int(b)) for a, b in zip(s, s[1:])]

for s in ["132", "129", "4657", "4690"]:
    j = jumps(s)
    print(s, j, max(j) <= 2)

132 [2, 1] True
129 [1, 7] False
4657 [2, 1, 2] True
4690 [2, 3, 9] False


## 5. Cách 1 — Brute Force / baseline

Làm đúng nghĩa đen của đề: **tính hiệu cho mọi cặp liền kề**, lưu vào một danh sách, rồi kiểm tra từng hiệu. Cách này tạo list phụ nhưng dễ đọc nhất:

1. Duyệt `i` từ `0` đến `n - 2`.
2. Tính `d = abs(int(s[i]) - int(s[i+1]))` và thêm vào list `diffs`.
3. Cuối cùng trả `True` nếu mọi `d` trong `diffs` đều `<= 2`.

Nút thắt: luôn tính **hết** `n - 1` hiệu dù đã có hiệu vi phạm từ đầu. Với `n <= 100` không đáng kể, nhưng nó gợi ý cải tiến: **dừng sớm**.

In [8]:
class SolutionBrute:
    def isAdjacentDiffAtMostTwo(self, s):
        diffs = []
        for i in range(len(s) - 1):
            diffs.append(abs(int(s[i]) - int(s[i + 1])))
        for d in diffs:
            if d > 2:
                return False
        return True

print(SolutionBrute().isAdjacentDiffAtMostTwo("132"), SolutionBrute().isAdjacentDiffAtMostTwo("129"))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho list `diffs`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** bài này là kiểu **"với mọi cặp thì điều kiện P đúng"**. Phủ định của nó là **"tồn tại một cặp làm P sai"**. Vì vậy:

- Không cần lưu danh sách hiệu; chỉ cần quét từ trái sang phải và **trả `False` ngay khi gặp một cặp vi phạm**;
- Quét hết mà không gặp vi phạm thì trả `True`.

**Quy tắc "nếu... thì...":**

- nếu `abs(s[i] - s[i+1]) > 2` thì kết quả `False` (kết thúc);
- nếu đã xét hết `n - 1` cặp mà chưa vi phạm thì kết quả `True`.

**Invariant:** sau khi xử lý `k` cặp đầu mà chưa dừng, ta biết chắc `k` cặp đó đều thỏa điều kiện. Chuỗi có đúng hai ký tự chỉ có đúng một cặp để xét.

## 7. Các cách tiếp theo

### Cách 2 — Vòng lặp thoát sớm (early exit)

Duyệt từng cặp, vi phạm là `return False` ngay.

In [9]:
class SolutionLoop:
    def isAdjacentDiffAtMostTwo(self, s):
        for i in range(len(s) - 1):
            if abs(int(s[i]) - int(s[i + 1])) > 2:
                return False
        return True

print(SolutionLoop().isAdjacentDiffAtMostTwo("132"), SolutionLoop().isAdjacentDiffAtMostTwo("129"))

True False


**Độ phức tạp:** Time `O(n)` (thoát sớm giúp best case `O(1)`), Space `O(1)`.

### Cách 3 — `zip` kết hợp `all`

`all(abs(int(a) - int(b)) <= 2 for a, b in zip(s, s[1:]))`. Gọn một dòng. Lưu ý `s[1:]` tạo chuỗi mới `O(n)` nhưng không đáng kể với `n <= 100`.

In [10]:
class SolutionZip:
    def isAdjacentDiffAtMostTwo(self, s):
        return all(abs(int(a) - int(b)) <= 2 for a, b in zip(s, s[1:]))

print(SolutionZip().isAdjacentDiffAtMostTwo("132"), SolutionZip().isAdjacentDiffAtMostTwo("129"))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (do lát cắt `s[1:]`; generator không lưu hiệu).

### Cách 4 — So sánh trực tiếp mã ASCII, không đổi sang `int`

Mã ASCII của chữ số liên tiếp (`'0' = 48`, `'1' = 49`...), nên `ord(a) - ord(b) = int(a) - int(b)`. Bỏ qua bước `int` để nhanh và tránh tạo đối tượng phụ.

In [11]:
class SolutionOrd:
    def isAdjacentDiffAtMostTwo(self, s):
        prev = ord(s[0])
        for ch in s[1:]:
            cur = ord(ch)
            if abs(cur - prev) > 2:
                return False
            prev = cur
        return True

print(SolutionOrd().isAdjacentDiffAtMostTwo("132"), SolutionOrd().isAdjacentDiffAtMostTwo("129"))

True False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `s[1:]` (hoặc `O(1)` nếu dùng chỉ số).

### Cách 5 — Bảng tra cứu các cặp hợp lệ

Chỉ có `10 x 10 = 100` cặp chữ số. Dựng trước **tập các cặp hợp lệ** `ok = {(a, b) : |a-b| <= 2}`, rồi mỗi cặp liền kề chỉ cần hỏi "có thuộc `ok` không". Cách này không phải tính toán lặp lại và tách rõ "luật" khỏi "việc duyệt".

In [12]:
DIGITS = "0123456789"
OK_PAIRS = {(a, b) for a in DIGITS for b in DIGITS if abs(int(a) - int(b)) <= 2}

class SolutionTable:
    def isAdjacentDiffAtMostTwo(self, s):
        return all(pair in OK_PAIRS for pair in zip(s, s[1:]))

print(len(OK_PAIRS), SolutionTable().isAdjacentDiffAtMostTwo("132"), SolutionTable().isAdjacentDiffAtMostTwo("129"))

44 True False


**Độ phức tạp:** Time `O(n)` (mỗi tra cứu `O(1)`), Space `O(1)` cho bảng cố định 100 cặp (+ `O(n)` lát cắt).

## 8. TRACE từng bước

Trace Cách 2 trên `"4657"` (hợp lệ) và `"4690"` (hỏng giữa chừng):

| Chuỗi | i | `s[i]`, `s[i+1]` | hiệu | hành động |
|---|---|---|---|---|
| `"4690"` | 0 | 4, 6 | 2 | tiếp tục |
| `"4690"` | 1 | 6, 9 | 3 | **return False** |

In [13]:
def trace_loop(s):
    print("s =", s)
    for i in range(len(s) - 1):
        a, b = int(s[i]), int(s[i + 1])
        d = abs(a - b)
        print(f"  i={i}: s[{i}]={a}, s[{i + 1}]={b}, hiệu={d}", end=" ")
        if d > 2:
            print("-> VI PHẠM, return False")
            return False
        print("-> ổn")
    print("  -> không có vi phạm, return True")
    return True

assert trace_loop("4657") is True
assert trace_loop("4690") is False
assert trace_loop("12") is True

s = 4657
  i=0: s[0]=4, s[1]=6, hiệu=2 -> ổn
  i=1: s[1]=6, s[2]=5, hiệu=1 -> ổn
  i=2: s[2]=5, s[3]=7, hiệu=2 -> ổn
  -> không có vi phạm, return True
s = 4690
  i=0: s[0]=4, s[1]=6, hiệu=2 -> ổn
  i=1: s[1]=6, s[2]=9, hiệu=3 -> VI PHẠM, return False
s = 12
  i=0: s[0]=1, s[1]=2, hiệu=1 -> ổn
  -> không có vi phạm, return True


In [14]:
def trace_table(s):
    # in bảng tra cứu cho từng cặp
    ok = lambda a, b: abs(int(a) - int(b)) <= 2
    for k, (a, b) in enumerate(zip(s, s[1:])):
        print(f"cặp #{k}: ({a},{b}) thuộc OK_PAIRS? {(a, b) in OK_PAIRS}  (tính trực tiếp: {ok(a, b)})")

trace_table("1329")

cặp #0: (1,3) thuộc OK_PAIRS? True  (tính trực tiếp: True)
cặp #1: (3,2) thuộc OK_PAIRS? True  (tính trực tiếp: True)
cặp #2: (2,9) thuộc OK_PAIRS? False  (tính trực tiếp: False)


## 9. Code nộp LeetCode

Dùng Cách 2 (vòng lặp thoát sớm, `O(1)` bộ nhớ phụ).

In [15]:
from typing import List, Optional


class Solution:
    def isAdjacentDiffAtMostTwo(self, s: str) -> bool:
        for i in range(len(s) - 1):
            if abs(int(s[i]) - int(s[i + 1])) > 2:
                return False
        return True


print(Solution().isAdjacentDiffAtMostTwo("132"), Solution().isAdjacentDiffAtMostTwo("129"))

True False


## 10. Vì sao thuật toán đúng

1. **Chỉ cần xét cặp liền kề.** Đề định nghĩa điều kiện theo từng cặp `(s[i], s[i+1])`; không có ràng buộc giữa các chữ số xa nhau.
2. **Quét đủ mọi cặp.** Vòng `for i in range(len(s) - 1)` đi qua `i = 0, 1, ..., n-2`, tức là đúng `n - 1` cặp liền kề, không thừa, không thiếu.
3. **Trả `False` đúng lúc.** Nếu có cặp mà `abs(...) > 2` thì điều kiện "mọi cặp đều `<= 2`" sai, nên trả `False` là chính xác.
4. **Trả `True` đúng lúc.** Nếu thoát khỏi vòng lặp mà chưa gặp vi phạm thì mọi cặp đều `<= 2` -> `True`.
5. **`abs` xử lý cả hai chiều.** Dù chữ số tăng hay giảm, hiệu tuyệt đối đo đúng khoảng cách.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`2 <= n <= 100`).

- Cách 1: Time `O(n)`, Space `O(n)` (list hiệu).
- Cách 2 (nộp): worst case `O(n)`, best case `O(1)` (vi phạm ngay cặp đầu), Space `O(1)`.
- Cách 3, 4: Time `O(n)`, Space `O(n)` do `s[1:]`.
- Cách 5: Time `O(n)`, Space `O(1)` bảng cố định.

Không thể nhanh hơn `O(n)` trong trường hợp xấu nhất vì chuỗi hợp lệ buộc phải xem **mọi** cặp (một cặp bị bỏ sót có thể là cặp vi phạm).

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [16]:
import random

solutions = [SolutionBrute(), SolutionLoop(), SolutionZip(), SolutionOrd(), SolutionTable(), Solution()]

fixed = [
    ("132", True), ("129", False), ("12", True), ("19", False), ("00", True),
    ("90", False), ("13", True), ("14", False), ("4657", True), ("4690", False),
    ("0123456789", True), ("02468", True), ("0369", False),
    ("9" * 100, True), ("98765432100123456789", True), ("0" * 99 + "9", False),
]
for s, expected in fixed:
    for sol in solutions:
        assert sol.isAdjacentDiffAtMostTwo(s) == expected, (type(sol).__name__, s)
print("test cố định OK:", len(fixed), "ca")

def oracle(s):
    return max(abs(int(a) - int(b)) for a, b in zip(s, s[1:])) <= 2

random.seed(3931)
cnt = {True: 0, False: 0}
for _ in range(600):
    n = random.randint(2, 100)
    if random.random() < 0.5:
        # chuỗi "đi bộ" quanh trục số: bước nhảy trong [-2, 2] nên đa phần hợp lệ
        d = random.randint(0, 9); digits = [d]
        for _ in range(n - 1):
            d = min(9, max(0, d + random.randint(-2, 2))); digits.append(d)
        s = "".join(map(str, digits))
        if random.random() < 0.3:
            i = random.randrange(n); s = s[:i] + random.choice("0123456789") + s[i + 1:]
    else:
        s = "".join(random.choice("0123456789") for _ in range(n))
    expected = oracle(s)
    cnt[expected] += 1
    for sol in solutions:
        assert sol.isAdjacentDiffAtMostTwo(s) == expected, (type(sol).__name__, s)
print("test ngẫu nhiên OK, phân bố kết quả:", cnt)
assert cnt[True] > 50 and cnt[False] > 50

test cố định OK: 16 ca
test ngẫu nhiên OK, phân bố kết quả: {True: 221, False: 379}


### Test thuộc tính (property test)

(1) Nếu `s` hợp lệ thì **mọi chuỗi con liên tiếp** của `s` cũng hợp lệ. (2) Đảo ngược `s` không đổi kết quả (hiệu tuyệt đối đối xứng). (3) Cộng cùng một số vào mọi chữ số (nếu vẫn trong `0..9`) không đổi kết quả.

In [17]:
random.seed(39310)
for _ in range(500):
    n = random.randint(2, 30)
    s = "".join(random.choice("0123456789") for _ in range(n))
    r = Solution().isAdjacentDiffAtMostTwo(s)
    assert Solution().isAdjacentDiffAtMostTwo(s[::-1]) == r
    if r:
        i = random.randint(0, n - 2); j = random.randint(i + 2, n)
        assert Solution().isAdjacentDiffAtMostTwo(s[i:j])
    mx = max(map(int, s))
    if mx < 9:
        shifted = "".join(str(int(c) + 1) for c in s)
        assert Solution().isAdjacentDiffAtMostTwo(shifted) == r
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `abs`

`int(a) - int(b)` có thể âm. `1 - 9 = -8` thì `-8 > 2` là `False` nên **không** bị coi là vi phạm, trong khi thực tế chênh lệch 8.

In [18]:
def wrong_no_abs(s):
    for i in range(len(s) - 1):
        if int(s[i]) - int(s[i + 1]) > 2:             # SAI: thiếu abs
            return False
    return True

print(wrong_no_abs("19"), "!=", Solution().isAdjacentDiffAtMostTwo("19"))
assert wrong_no_abs("19") is True and Solution().isAdjacentDiffAtMostTwo("19") is False
assert wrong_no_abs("91") is False        # chỉ bắt được khi chữ số GIẢM

True != False


### Lỗi 2: nhầm `< 2` với `<= 2` (hoặc `> 2` thành `>= 2`)

Đề cho phép hiệu **đúng bằng 2**. Viết điều kiện hợp lệ là `< 2` làm `"13"` (hiệu 2) bị loại nhầm.

In [19]:
def wrong_strict(s):
    return all(abs(int(a) - int(b)) < 2 for a, b in zip(s, s[1:]))   # SAI: phải <= 2

print(wrong_strict("13"), "!=", Solution().isAdjacentDiffAtMostTwo("13"))
assert wrong_strict("13") is False and Solution().isAdjacentDiffAtMostTwo("13") is True
assert wrong_strict("132") is False       # ví dụ 1 của đề cũng sai!

False != True


### Lỗi 3: lệch chỉ số, vòng lặp chạy tới hết chuỗi

`range(len(s))` làm `s[i + 1]` vượt biên ở lần cuối -> `IndexError`. Phải dùng `range(len(s) - 1)`.

In [20]:
def wrong_range(s):
    for i in range(len(s)):                            # SAI: phải len(s) - 1
        if abs(int(s[i]) - int(s[i + 1])) > 2:
            return False
    return True

try:
    wrong_range("132")
except IndexError as e:
    print("IndexError:", e)
assert Solution().isAdjacentDiffAtMostTwo("132") is True

IndexError: string index out of range


### Lỗi 4: chỉ kiểm tra cặp đầu hoặc cặp cuối

Đề yêu cầu **mọi** cặp. Chỉ xét `s[0], s[1]` bỏ lọt vi phạm ở giữa.

In [21]:
def wrong_first_only(s):
    return abs(int(s[0]) - int(s[1])) <= 2            # SAI: bỏ các cặp còn lại

print(wrong_first_only("129"), "!=", Solution().isAdjacentDiffAtMostTwo("129"))
assert wrong_first_only("129") is True and Solution().isAdjacentDiffAtMostTwo("129") is False

True != False


### Lỗi 5: trừ trực tiếp hai ký tự

`'3' - '1'` không hợp lệ trong Python (`TypeError`). Cần đổi sang số bằng `int(...)` hoặc `ord(...)`.

In [22]:
def wrong_str_minus(s):
    return all(abs(a - b) <= 2 for a, b in zip(s, s[1:]))   # SAI: a, b là chuỗi

try:
    wrong_str_minus("132")
except TypeError as e:
    print("TypeError:", e)
assert Solution().isAdjacentDiffAtMostTwo("132") is True

TypeError: unsupported operand type(s) for -: 'str' and 'str'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi cặp liền kề thỏa P" -> `all(P(a, b) for a, b in zip(s, s[1:]))`. Đây là mẫu xuất hiện rất nhiều (kiểm tra mảng tăng dần, kiểm tra chuỗi xen kẽ, ...).
- "Có tồn tại" -> `any`; "mọi" -> `all`; đừng quên `all([])` là `True` (chuỗi 1 ký tự không có cặp nào).
- `ord(a) - ord(b)` bỏ được việc gọi `int`; `abs` luôn đi kèm khi đề nói "hiệu tuyệt đối".
- Chuỗi chữ số: chỉ có 10 ký tự nên đôi khi có thể **dựng bảng** `10 x 10` thay vì tính lại.
- Đọc constraints: `n <= 100` nên `O(n)` hay thậm chí `O(n^2)` đều dư sức; hãy ưu tiên **code dễ đúng**.
- Luôn thử ba chuỗi: độ dài tối thiểu (`"12"`), vi phạm ở **cặp đầu**, vi phạm ở **cặp cuối**.

## 15. Biến thể của bài

1. Đổi ngưỡng `2` thành tham số `k`: `isAdjacentDiffAtMost(s, k)`.
2. Hỏi **đếm** số cặp liền kề vi phạm, hoặc trả về **vị trí đầu tiên** vi phạm.
3. Cho phép **xóa tối đa một** ký tự để chuỗi hợp lệ (cần xét kỹ hơn: kiểm tra cặp `(s[i-1], s[i+1])`).
4. Hiệu **vòng** (hai chữ số `9` và `0` coi như cách nhau 1, như trên mặt đồng hồ).
5. Mảng số thay cho chuỗi chữ số: **Monotonic Array** (896), **Special Array I** (3151: hai phần tử liền kề khác tính chẵn lẻ).
6. Kiểm tra chuỗi **xen kẽ** (không có hai ký tự giống nhau liền kề): bài tương tự về mặt kỹ thuật.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute (lưu hiệu) | tính mọi hiệu rồi kiểm tra | `O(n)` | `O(n)` | Học, gỡ lỗi (in được list hiệu) |
| 2. Vòng lặp thoát sớm | `return False` khi vi phạm | `O(n)` (best `O(1)`) | `O(1)` | **Nộp bài** |
| 3. `zip + all` | biểu thức một dòng | `O(n)` | `O(n)` | Code ngắn, dễ đọc |
| 4. `ord` | bỏ `int(...)` | `O(n)` | `O(1)`/`O(n)` | Khi muốn nhanh nhất có thể |
| 5. Bảng cặp hợp lệ | tra `set` 100 cặp | `O(n)` | `O(1)` | Khi luật phức tạp, muốn tách luật khỏi vòng lặp |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().isAdjacentDiffAtMostTwo("0246")` và `Solution().isAdjacentDiffAtMostTwo("0235")` ra gì?

**Bài 2 (tính tay):** liệt kê các hiệu liền kề của `"86420"` rồi cho biết kết quả; đổi một chữ số để chuỗi chắc chắn thành `False`.

**Bài 3 (điền TODO):** hoàn thành `first_bad_index(s)`: trả về **chỉ số `i` đầu tiên** mà `abs(s[i] - s[i+1]) > 2`, hoặc `-1` nếu không có.

**Bài 4 (tự viết lại):** viết `count_bad_pairs(s)` đếm số cặp liền kề vi phạm, rồi `is_good(s)` dựa vào nó (`count_bad_pairs(s) == 0`).

**Bài 5 (mini challenge):** viết `is_adjacent_diff_at_most_k_circular(s, k)` với khoảng cách **vòng** giữa hai chữ số: `min(d, 10 - d)` (với `d = abs(a - b)`).

In [23]:
# Bài 1: đoán rồi chạy
print(Solution().isAdjacentDiffAtMostTwo("0246"), Solution().isAdjacentDiffAtMostTwo("0235"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def first_bad_index_todo(s):
    # TODO: duyệt i từ 0 đến len(s) - 2, trả về i đầu tiên có abs(int(s[i]) - int(s[i+1])) > 2, hoặc -1
    return -1

True True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `"0246"` có các bước nhảy toàn bằng 2. `"0235"`: `0->2` là 2, `2->3` là 1, `3->5` là 2. Bài 2: viết 4 hiệu cho 5 chữ số.

**Mức 2 (vừa):** Bài 3: vòng `for i in range(len(s) - 1)`; gặp vi phạm thì `return i`; hết vòng `return -1`. Bài 4: `sum(1 for a, b in zip(s, s[1:]) if abs(int(a) - int(b)) > 2)`. Bài 5: tính `d = abs(a - b)` rồi so `min(d, 10 - d) <= k`.

**Mức 3 (gần đáp án):** Bài 1: cả hai `True`. Bài 2: hiệu `2,2,2,2` -> `True`; đổi thành `"86429"` -> `False`. Bài 3: `"1329" -> 2`, `"132" -> -1`. Bài 5: với `k = 2`, `"09"` hợp lệ (khoảng cách vòng là 1).

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().isAdjacentDiffAtMostTwo("0246") is True and Solution().isAdjacentDiffAtMostTwo("0235") is True
assert jumps("86420") == [2, 2, 2, 2] and Solution().isAdjacentDiffAtMostTwo("86420") is True
assert Solution().isAdjacentDiffAtMostTwo("86429") is False

# Bài 3
def first_bad_index(s):
    for i in range(len(s) - 1):
        if abs(int(s[i]) - int(s[i + 1])) > 2:
            return i
    return -1

assert first_bad_index("1329") == 2 and first_bad_index("132") == -1 and first_bad_index("19") == 0

# Bài 4
def count_bad_pairs(s):
    return sum(1 for a, b in zip(s, s[1:]) if abs(int(a) - int(b)) > 2)

def is_good(s):
    return count_bad_pairs(s) == 0

assert count_bad_pairs("0909") == 3 and is_good("132") and not is_good("129")

# Bài 5
def is_adjacent_diff_at_most_k_circular(s, k):
    for a, b in zip(s, s[1:]):
        d = abs(int(a) - int(b))
        if min(d, 10 - d) > k:
            return False
    return True

assert is_adjacent_diff_at_most_k_circular("09", 2) is True
assert is_adjacent_diff_at_most_k_circular("05", 2) is False and is_adjacent_diff_at_most_k_circular("0", 0) is True
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def isAdjacentDiffAtMostTwo(self, s: str) -> bool:
        for i in range(len(s) - 1):            # n - 1 cặp liền kề
            if abs(int(s[i]) - int(s[i + 1])) > 2:
                return False                    # tìm thấy cặp vi phạm
        return True                             # không có cặp nào vi phạm
```

Khung chung cho mọi bài "mọi cặp liền kề thỏa P": `all(P(a, b) for a, b in zip(s, s[1:]))`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Có `abs`, điều kiện là `<= 2` (không phải `< 2`), vòng lặp `range(len(s) - 1)`.
- [ ] Đã thử `"12"` (độ dài nhỏ nhất), vi phạm ở cặp đầu và ở cặp cuối.